# CS 521 HW3, Problem 3: Multi-Norm Adversarial Training

Use Google Colab with a GPU: **Runtime > Change runtime type > T4 GPU**.

All the code is in this notebook. Complete each place that has a `TODO` and `raise NotImplementedError`.
Do not change the data, the model, the attacks, the evaluation or the default hyperparameters,
unless the homework tells you to change them.

**Threat model.** For each test image $x$, the perturbation region is
$B_\infty(x, 8/255) \cup B_2(x, 0.5) \cup B_1(x, 12)$.
The union accuracy is the percentage of test images that are robust against all three attacks simultaneously.

# Set up for dataset and model

Download the base model, a subset of 10,000 CIFAR-10 training images (1,000 for each class),
and the first 1,000 CIFAR-10 test images from Google Drive.

In [1]:
!gdown 17vWkPpVRFUYAxWv3_NEj1gV7Lxh-Cnes -O linf_base.pt
!gdown 1LoY_o51zNQkrkfvAlGE-44g9G-c38OJe -O cifar10_train_10k.pt
!gdown 1bElxIZ6A708IOGaTMcDq_AsCpC9QQseA -O cifar10_test_1k.pt

/bin/bash: line 1: gdown: command not found
/bin/bash: line 1: gdown: command not found
/bin/bash: line 1: gdown: command not found


In [2]:
import time

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

NORMS = ("linf", "l2", "l1")
EPS = {"linf": 8 / 255, "l2": 0.5, "l1": 12.0}

device: cuda


In [3]:
def load_data(path):
    # The files store images as uint8. Convert them to floats in [0, 1] on the GPU.
    x, y = torch.load(path)
    return x.float().div(255).to(device), y.to(device)


# Training data: 10,000 images. Test data: the first 1,000 CIFAR-10 test images.
x_train, y_train = load_data("cifar10_train_10k.pt")
x_test, y_test = load_data("cifar10_test_1k.pt")
print("train:", tuple(x_train.shape), "test:", tuple(x_test.shape))


def augment(x):
    # Random horizontal flip and random crop (padding 4) for each image.
    n = x.shape[0]
    flip = torch.rand(n, device=x.device) < 0.5
    x = torch.where(flip.view(-1, 1, 1, 1), x.flip(3), x)
    padded = F.pad(x, (4, 4, 4, 4))
    i = torch.randint(0, 9, (n, 1, 1), device=x.device) + torch.arange(32, device=x.device).view(1, 32, 1)
    j = torch.randint(0, 9, (n, 1, 1), device=x.device) + torch.arange(32, device=x.device).view(1, 1, 32)
    return padded[torch.arange(n, device=x.device).view(n, 1, 1), :, i, j].permute(0, 3, 1, 2).contiguous()


def batches(x, y, batch_size, shuffle):
    idx = torch.randperm(len(x), device=x.device) if shuffle else torch.arange(len(x), device=x.device)
    for k in range(0, len(x), batch_size):
        j = idx[k:k + batch_size]
        yield x[j], y[j]

train: (10000, 3, 32, 32) test: (1000, 3, 32, 32)


In [4]:
# PreActResNet-18 with half width (2.8M parameters). The input is an image in [0, 1].
class PreActBlock(nn.Module):
    def __init__(self, in_planes, planes, stride):
        super().__init__()
        self.bn1 = nn.BatchNorm2d(in_planes)
        self.conv1 = nn.Conv2d(in_planes, planes, 3, stride, 1, bias=False)
        self.bn2 = nn.BatchNorm2d(planes)
        self.conv2 = nn.Conv2d(planes, planes, 3, 1, 1, bias=False)
        self.shortcut = None
        if stride != 1 or in_planes != planes:
            self.shortcut = nn.Conv2d(in_planes, planes, 1, stride, bias=False)

    def forward(self, x):
        out = F.relu(self.bn1(x))
        shortcut = self.shortcut(out) if self.shortcut is not None else x
        out = self.conv1(out)
        out = self.conv2(F.relu(self.bn2(out)))
        return out + shortcut


class PreActResNet18(nn.Module):
    def __init__(self, width=32, num_classes=10):
        super().__init__()
        self.register_buffer("mean", torch.tensor([0.4914, 0.4822, 0.4465]).view(1, 3, 1, 1))
        self.register_buffer("std", torch.tensor([0.2471, 0.2435, 0.2616]).view(1, 3, 1, 1))
        self.conv1 = nn.Conv2d(3, width, 3, 1, 1, bias=False)
        layers, in_planes = [], width
        for mult, stride in [(1, 1), (1, 1), (2, 2), (2, 1), (4, 2), (4, 1), (8, 2), (8, 1)]:
            layers.append(PreActBlock(in_planes, mult * width, stride))
            in_planes = mult * width
        self.layers = nn.Sequential(*layers)
        self.bn = nn.BatchNorm2d(in_planes)
        self.linear = nn.Linear(in_planes, num_classes)

    def forward(self, x):
        out = self.layers(self.conv1((x - self.mean) / self.std))
        out = F.adaptive_avg_pool2d(F.relu(self.bn(out)), 1).flatten(1)
        return self.linear(out)


def make_model():
    return PreActResNet18().to(device)


def load_base_model():
    model = make_model()
    model.load_state_dict(torch.load("linf_base.pt", map_location=device))
    return model

# Implement the Attacks

Each attack is untargeted, uses the cross-entropy loss, and returns `x_adv` with
$\|x_{adv} - x\|_p \leq \epsilon$ and $x_{adv} \in [0, 1]$. The attacks must not change the mode (`train` or `eval`) of the model.

* `pgd_linf`: start at a uniformly random point in the $\ell_\infty$ ball. For each step, add `alpha * sign(grad)`,
  clip the perturbation to $[-\epsilon, \epsilon]$, and clip the image to $[0, 1]$.
* `pgd_l2`: start at a random point in the $\ell_2$ ball. For each step, add `alpha * grad / ||grad||_2`,
  scale the perturbation back to the $\ell_2$ ball if its norm is larger than $\epsilon$, and clip the image to $[0, 1]$.

You can use your code from HW1. The $\ell_1$ attack (sparse $\ell_1$ descent, Tramer & Boneh 2019) is complete.

In [5]:
def clip_box(x, delta):
    # Make x + delta an image in [0, 1]. This never increases |delta_i|.
    return (x + delta).clamp(0, 1) - x


def input_grad(model, x, delta, y):
    delta = delta.detach().requires_grad_(True)
    loss = F.cross_entropy(model(x + delta), y, reduction="sum")
    (grad,) = torch.autograd.grad(loss, delta)
    return grad.detach()


def l2_norm(v):
    return v.flatten(1).norm(dim=1).view(-1, 1, 1, 1)


def pgd_linf(model, x, y, eps, alpha, steps, rand_init=True):
    delta = torch.zeros_like(x)
    if rand_init:
        delta = torch.empty_like(x).uniform_(-eps, eps)
        delta = clip_box(x, delta)
        
    for _ in range(steps):
        grad = input_grad(model, x, delta, y)
        delta = delta + alpha * grad.sign()
        delta = torch.clamp(delta, -eps, eps)
        delta = clip_box(x, delta)
        
    return (x + delta).detach()


def pgd_l2(model, x, y, eps, alpha, steps, rand_init=True):
    delta = torch.zeros_like(x)
    if rand_init:
        delta = torch.randn_like(x)
        delta_norm = l2_norm(delta).clamp_min(1e-12)
        r = torch.empty(x.shape[0], 1, 1, 1, device=x.device).uniform_(0, eps)
        delta = delta / delta_norm * r
        delta = clip_box(x, delta)
        
    for _ in range(steps):
        grad = input_grad(model, x, delta, y)
        grad_norm = l2_norm(grad).clamp_min(1e-12)
        delta = delta + alpha * grad / grad_norm
        
        delta_norm = l2_norm(delta).clamp_min(1e-12)
        delta = torch.where(delta_norm > eps, delta * (eps / delta_norm), delta)
        delta = clip_box(x, delta)
        
    return (x + delta).detach()

In [6]:
def project_l1(delta, eps):
    # Projection of each image of delta onto the L1 ball of radius eps (Duchi et al., 2008).
    v = delta.flatten(1)
    a = v.abs()
    outside = a.sum(dim=1) > eps
    if not outside.any():
        return delta
    mu, _ = a.sort(dim=1, descending=True)
    cumsum = mu.cumsum(dim=1)
    idx = torch.arange(1, v.shape[1] + 1, device=v.device, dtype=v.dtype)
    rho = ((mu - (cumsum - eps) / idx > 0) * idx).argmax(dim=1, keepdim=True)
    theta = (cumsum.gather(1, rho) - eps) / (rho + 1).to(v.dtype)
    w = (a - theta).clamp_min(0) * v.sign()
    return torch.where(outside.unsqueeze(1), w, v).view(delta.shape)


def pgd_l1(model, x, y, eps, alpha, steps, rand_init=True, k=20):
    # Each step changes only the k coordinates with the largest gradient.
    # Coordinates at the image boundary that point outward are skipped.
    delta = torch.zeros_like(x)
    if rand_init:
        delta = torch.distributions.Laplace(0.0, 1.0).sample(x.shape).to(x.device)
        r = torch.rand(x.shape[0], 1, 1, 1, device=x.device)
        delta = delta / delta.flatten(1).abs().sum(1).view(-1, 1, 1, 1) * eps * r
    delta = clip_box(x, delta)
    for _ in range(steps):
        grad = input_grad(model, x, delta, y).flatten(1)
        x_cur = (x + delta).flatten(1)
        blocked = ((grad > 0) & (x_cur >= 1)) | ((grad < 0) & (x_cur <= 0))
        grad = grad.masked_fill(blocked, 0)
        top = grad.abs().topk(k, dim=1).indices
        step = torch.zeros_like(grad).scatter(1, top, grad.gather(1, top).sign())
        delta = clip_box(x, delta + alpha * (step / k).view_as(x))
        delta = project_l1(delta, eps)
    return (x + delta).detach()


ATTACKS = {"linf": pgd_linf, "l2": pgd_l2, "l1": pgd_l1}


def attack(model, x, y, norm, steps=10):
    # The attack for one norm with the default eps and step size.
    alpha = EPS["l1"] / steps ** 0.5 if norm == "l1" else 2.5 * EPS[norm] / steps
    return ATTACKS[norm](model, x, y, EPS[norm], alpha, steps)

# Evaluate Single and Multi-Norm Robust Accuracy

`test_model` gives the clean accuracy, the robust accuracy for each norm, and the union accuracy (in %) on the
1,000 test images. An image is robust for a norm if the model is correct on the clean image and on the result
of each restart of the attack. An image is robust for the union if it is robust for all three norms simultaneously.
Use the default settings (`steps=50, restarts=2`) for all the numbers in your report.

In [7]:
def test_model(model, steps=50, restarts=2, batch_size=250):
    model.eval()
    torch.manual_seed(0)  # the same random starts of the attacks for each model, so the results are comparable
    start = time.time()
    correct = {k: 0 for k in ("clean", *NORMS, "union")}
    for i in range(0, len(x_test), batch_size):
        x, y = x_test[i:i + batch_size], y_test[i:i + batch_size]
        with torch.no_grad():
            clean = model(x).argmax(1) == y
        union = clean.clone()
        for norm in NORMS:
            robust = clean.clone()
            for _ in range(restarts):
                x_adv = attack(model, x, y, norm, steps)
                with torch.no_grad():
                    robust &= model(x_adv).argmax(1) == y
            correct[norm] += robust.sum().item()
            union &= robust
        correct["clean"] += clean.sum().item()
        correct["union"] += union.sum().item()
    acc = {k: 100.0 * v / len(y_test) for k, v in correct.items()}
    print("  ".join(f"{k} {v:.1f}" for k, v in acc.items()) + f"   ({time.time() - start:.0f} s)")
    return acc

In [8]:
# The base model: L-infinity PGD adversarial training for 30 epochs on all 50,000 training images.
test_model(load_base_model())

clean 78.7  linf 48.3  l2 59.9  l1 23.6  union 23.5   (33 s)


{'clean': 78.7, 'linf': 48.3, 'l2': 59.9, 'l1': 23.6, 'union': 23.5}

# Adversarial Training

`adv_train(model, loss_fn, ...)` fine-tunes `model` for `epochs` epochs on the training subset.
For each batch, it calls `loss_fn(model, x, y, **kwargs)` and does one optimizer step on the loss.

A loss function makes the adversarial examples itself. Put the model in `eval` mode for the attack
(so that the attack does not change the BatchNorm statistics), and in `train` mode before you calculate the loss.

If `gp_beta` is set, `adv_train` does the gradient projection of RAMP (part (b)) before the optimizer step.

In [8]:
def adv_train(model, loss_fn, epochs=10, lr=0.01, batch_size=128, gp_beta=None, seed=0, **kwargs):
    torch.manual_seed(seed)
    opt = torch.optim.SGD(model.parameters(), lr=lr, momentum=0.9, weight_decay=5e-4)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, epochs * (len(x_train) // batch_size))
    params = list(model.parameters())
    start = time.time()
    for epoch in range(epochs):
        losses = []
        for x, y in batches(x_train, y_train, batch_size, shuffle=True):
            if len(y) < batch_size:  # skip the last, smaller batch
                continue
            x = augment(x)
            loss = loss_fn(model, x, y, **kwargs)
            opt.zero_grad()
            if gp_beta is None:
                loss.backward()
            else:
                # RAMP (part (b)): combine the adversarial gradient g_a and the natural gradient g_n
                # for each parameter tensor. gradient_projection is defined in part (b).
                g_a = torch.autograd.grad(loss, params)
                g_n = torch.autograd.grad(F.cross_entropy(model(x), y), params)
                for p, ga, gn in zip(params, g_a, g_n):
                    p.grad = gp_beta * gradient_projection(gn, ga) + (1 - gp_beta) * ga
            opt.step()
            sched.step()
            losses.append(loss.item())
        print(f"epoch {epoch + 1}: loss {np.mean(losses):.3f} ({time.time() - start:.0f} s)")
    model.eval()
    return model

In [11]:
# Problem 1A

def simple_cross_entropy_loss(model, x, y):
    model.train()
    return F.cross_entropy(model(x), y)

epsilons_to_test = [0.01, 0.02, 0.04, 0.06]
og_eps = EPS["linf"]

print("Problem #1A Code")

standard_model = adv_train(make_model(), simple_cross_entropy_loss, epochs=5)
adv_model = adv_train(make_model(), single_norm_loss, epochs=5, norm="linf")

for eps in epsilons_to_test:
    print("")
    print(f"linf epsilon = {eps}")
    EPS["linf"] = eps
    
    print("")
    print("Standard Model Evaluation:")
    test_model(standard_model)
    
    print("")
    print("Adversarial Model Evaluation:")
    test_model(adv_model)

EPS["linf"] = og_eps



Problem #1A Code
epoch 1: loss 1.875 (2 s)
epoch 2: loss 1.544 (5 s)
epoch 3: loss 1.379 (7 s)
epoch 4: loss 1.223 (10 s)
epoch 5: loss 1.129 (12 s)
epoch 1: loss 2.193 (13 s)
epoch 2: loss 2.104 (26 s)
epoch 3: loss 2.056 (39 s)
epoch 4: loss 2.033 (52 s)
epoch 5: loss 2.019 (65 s)

linf epsilon = 0.01

Standard Model Evaluation:
clean 57.6  linf 13.3  l2 6.3  l1 1.3  union 1.3   (31 s)

Adversarial Model Evaluation:
clean 37.4  linf 32.6  l2 30.4  l1 21.1  union 20.9   (30 s)

linf epsilon = 0.02

Standard Model Evaluation:
clean 57.6  linf 1.4  l2 6.3  l1 1.3  union 0.9   (30 s)

Adversarial Model Evaluation:
clean 37.4  linf 27.9  l2 30.4  l1 21.1  union 20.7   (30 s)

linf epsilon = 0.04

Standard Model Evaluation:
clean 57.6  linf 0.0  l2 6.3  l1 1.3  union 0.0   (30 s)

Adversarial Model Evaluation:
clean 37.4  linf 20.3  l2 30.4  l1 21.1  union 18.1   (30 s)

linf epsilon = 0.06

Standard Model Evaluation:
clean 57.6  linf 0.0  l2 6.3  l1 1.3  union 0.0   (30 s)

Adversarial Mo

In [12]:
# Problem 1B

def evaluate_attacks(model, epsilons, batch_size=250):
    model.eval()
    results = {"clean": 0, "fgsm": {eps: 0 for eps in epsilons}, "pgd": {eps: 0 for eps in epsilons}}
    
    for i in range(0, len(x_test), batch_size):
        x, y = x_test[i:i + batch_size], y_test[i:i + batch_size]
        
        # clean accuracy
        with torch.no_grad():
            clean_correct = (model(x).argmax(1) == y)
        results["clean"] += clean_correct.sum().item()
        
        for eps in epsilons:
            # fgsm attack
            x_fgsm = pgd_linf(model, x, y, eps=eps, alpha=eps, steps=1, rand_init=False)
            with torch.no_grad():
                results["fgsm"][eps] += (model(x_fgsm).argmax(1) == y).sum().item()
                
            # pgd attack
            steps = 10
            alpha = 2.5 * eps / steps
            x_pgd = pgd_linf(model, x, y, eps=eps, alpha=alpha, steps=steps, rand_init=True)
            with torch.no_grad():
                results["pgd"][eps] += (model(x_pgd).argmax(1) == y).sum().item()
                
    total = len(y_test)
    print(f"Clean Accuracy: {100.0 * results['clean'] / total:.1f}%")
    for eps in epsilons:
        fgsm_acc = 100.0 * results["fgsm"][eps] / total
        pgd_acc = 100.0 * results["pgd"][eps] / total
        print(f"Eps: {eps:<6} | FGSM Acc: {fgsm_acc:>5.1f}% | PGD Acc: {pgd_acc:>5.1f}%")


epsilons_to_test = [0.01, 0.02, 0.04, 0.06]

print("Problem #1B Code")

print("")
print("Standard Model Evaluation")
evaluate_attacks(standard_model, epsilons_to_test)

print("")
print("Adversarial Model Evaluation")
evaluate_attacks(adv_model, epsilons_to_test)

Problem #1B Code

Standard Model Evaluation
Clean Accuracy: 57.6%
Eps: 0.01   | FGSM Acc:  19.5% | PGD Acc:  13.7%
Eps: 0.02   | FGSM Acc:   6.7% | PGD Acc:   1.5%
Eps: 0.04   | FGSM Acc:   1.0% | PGD Acc:   0.0%
Eps: 0.06   | FGSM Acc:   0.3% | PGD Acc:   0.0%

Adversarial Model Evaluation
Clean Accuracy: 37.4%
Eps: 0.01   | FGSM Acc:  32.8% | PGD Acc:  32.8%
Eps: 0.02   | FGSM Acc:  28.4% | PGD Acc:  28.1%
Eps: 0.04   | FGSM Acc:  21.1% | PGD Acc:  20.6%
Eps: 0.06   | FGSM Acc:  15.4% | PGD Acc:  13.9%


# Part (a): Single-Norm Adversarial Training

Complete `single_norm_loss`: attack the batch with the norm `norm` (use `attack`), and return the
cross-entropy loss of the model on the adversarial images.

In [10]:
def single_norm_loss(model, x, y, norm):
    model.eval()
    x_adv = attack(model, x, y, norm)
    model.train()
    
    logits = model(x_adv)
    return F.cross_entropy(logits, y)

In [27]:
test_model(load_base_model())

clean 78.7  linf 48.3  l2 59.9  l1 23.6  union 23.5   (30 s)


{'clean': 78.7, 'linf': 48.3, 'l2': 59.9, 'l1': 23.6, 'union': 23.5}

In [11]:
model_linf = adv_train(load_base_model(), single_norm_loss, norm="linf")
test_model(model_linf)

epoch 1: loss 1.203 (13 s)
epoch 2: loss 1.180 (26 s)
epoch 3: loss 1.151 (39 s)
epoch 4: loss 1.126 (52 s)
epoch 5: loss 1.096 (65 s)
epoch 6: loss 1.067 (78 s)
epoch 7: loss 1.049 (91 s)
epoch 8: loss 1.030 (104 s)
epoch 9: loss 1.013 (116 s)
epoch 10: loss 1.008 (129 s)
clean 78.6  linf 44.6  l2 57.0  l1 20.3  union 20.2   (30 s)


{'clean': 78.6, 'linf': 44.6, 'l2': 57.0, 'l1': 20.3, 'union': 20.2}

In [12]:
model_l2 = adv_train(load_base_model(), single_norm_loss, norm="l2")
test_model(model_l2)

epoch 1: loss 0.897 (13 s)
epoch 2: loss 0.826 (26 s)
epoch 3: loss 0.787 (39 s)
epoch 4: loss 0.755 (52 s)
epoch 5: loss 0.715 (65 s)
epoch 6: loss 0.674 (78 s)
epoch 7: loss 0.631 (90 s)
epoch 8: loss 0.610 (103 s)
epoch 9: loss 0.591 (116 s)
epoch 10: loss 0.583 (129 s)
clean 82.0  linf 31.9  l2 60.1  l1 28.0  union 24.5   (30 s)


{'clean': 82.0, 'linf': 31.9, 'l2': 60.1, 'l1': 28.0, 'union': 24.5}

In [13]:
model_l1 = adv_train(load_base_model(), single_norm_loss, norm="l1")
test_model(model_l1)

epoch 1: loss 1.229 (20 s)
epoch 2: loss 0.822 (41 s)
epoch 3: loss 0.565 (62 s)
epoch 4: loss 0.451 (83 s)
epoch 5: loss 0.345 (104 s)
epoch 6: loss 0.280 (125 s)
epoch 7: loss 0.224 (147 s)
epoch 8: loss 0.200 (168 s)
epoch 9: loss 0.171 (189 s)
epoch 10: loss 0.171 (210 s)
clean 83.6  linf 0.0  l2 0.0  l1 24.7  union 0.0   (30 s)


{'clean': 83.6, 'linf': 0.0, 'l2': 0.0, 'l1': 24.7, 'union': 0.0}

# Part (b): Multi-Norm Adversarial Training (RAMP)

* `ramp_loss`: attack with $q = \ell_\infty$ and $r = \ell_1$. The loss is the mean of
  $\max(\mathcal{L}_{CE}(x^q), \mathcal{L}_{CE}(x^r))$ plus `lam` times $\mathcal{L}_{KL} = KL(p_q \| p_r)$.
  Calculate $\mathcal{L}_{KL}$ only on the images that the model classifies correctly under the $\ell_q$ attack.
* `gradient_projection(g_n, g_a)`: $\cos(g_n, g_a) \cdot g_n$ if $\cos(g_n, g_a) > 0$, else $0$.

`adv_train(..., gp_beta=0.5)` then uses `p.grad = gp_beta * GP(g_n, g_a) + (1 - gp_beta) * g_a` for each parameter
tensor, where $g_a$ is the gradient of `ramp_loss` and $g_n$ is the gradient of the cross-entropy loss on the clean images.

In [21]:
def ramp_loss(model, x, y, lam=0.5):
    # TODO: in eval mode, attack with linf and l1; in train mode, return the max of the two losses plus lam * KL
    model.eval()
    x_inf = attack(model, x, y, "linf")
    x_1 = attack(model, x, y, "l1")
    
    with torch.no_grad():
        correct_inf = model(x_inf).argmax(dim=1) == y
        
    model.train()
    logits_inf = model(x_inf)
    logits_1 = model(x_1)
    
    loss_inf = F.cross_entropy(logits_inf, y, reduction='none')
    loss_1 = F.cross_entropy(logits_1, y, reduction='none')
    loss_ce = torch.max(loss_inf, loss_1).mean()
    
    # KL divergence calc
    if correct_inf.any():
        log_p_1 = F.log_softmax(logits_1[correct_inf], dim=1)
        p_inf = F.softmax(logits_inf[correct_inf], dim=1)
        loss_kl = F.kl_div(log_p_1, p_inf, reduction='batchmean')
    else:
        loss_kl = torch.tensor(0.0, device=x.device)
        
    return loss_ce + lam * loss_kl


def gradient_projection(g_n, g_a):
    gn_flat = g_n.flatten()
    ga_flat = g_a.flatten()
    
    # cosine similarity func (angle of vectors is dot product)
    cosGrad = torch.dot(gn_flat, ga_flat) / (torch.norm(gn_flat) * torch.norm(ga_flat) + 1e-12)
    
    if cosGrad > 0:
        return cosGrad * g_n
    else:
        return torch.zeros_like(g_n)

In [22]:
model_ramp = adv_train(load_base_model(), ramp_loss, gp_beta=0.5)
test_model(model_ramp)

epoch 1: loss 1.431 (32 s)
epoch 2: loss 1.330 (65 s)
epoch 3: loss 1.303 (97 s)
epoch 4: loss 1.280 (130 s)
epoch 5: loss 1.254 (162 s)
epoch 6: loss 1.227 (196 s)
epoch 7: loss 1.198 (229 s)
epoch 8: loss 1.186 (261 s)
epoch 9: loss 1.171 (294 s)
epoch 10: loss 1.166 (327 s)
clean 79.2  linf 41.1  l2 61.5  l1 45.0  union 39.3   (30 s)


{'clean': 79.2, 'linf': 41.1, 'l2': 61.5, 'l1': 45.0, 'union': 39.3}

# Part (c): Your Method

Write the training loss of your method in `my_loss`, and train with `adv_train(load_base_model(), my_loss)`.
If your method needs a different update (not the gradient of one loss), write your own copy of `adv_train`.
Run your method and RAMP with `seed=0` and `seed=1`.
Then evaluate your best model again with stronger attacks: `test_model(model, steps=200, restarts=5)`.

In [25]:
# My method: calc all three losses for linf, l2, l1 and do a weighted sum via softmax over the losses

def my_loss(model, x, y):
    model.eval()
    x_linf = attack(model, x, y, "linf")
    x_l2 = attack(model, x, y, "l2")
    x_l1 = attack(model, x, y, "l1")

    model.train()
    logits_linf = model(x_linf)
    logits_l2 = model(x_l2)
    logits_l1 = model(x_l1)

    loss_linf = F.cross_entropy(logits_linf, y)
    loss_l2 = F.cross_entropy(logits_l2, y)
    loss_l1 = F.cross_entropy(logits_l1, y)

    losses = torch.stack([loss_linf, loss_l2, loss_l1])
    weights = F.softmax(losses.detach(), dim=0)

    final_loss = weights[0] * loss_linf + weights[1] * loss_l2 + weights[2] * loss_l1
    return final_loss

In [26]:
model_mine = adv_train(load_base_model(), my_loss)
test_model(model_mine)

epoch 1: loss 1.224 (40 s)
epoch 2: loss 1.145 (80 s)
epoch 3: loss 1.105 (121 s)
epoch 4: loss 1.072 (161 s)
epoch 5: loss 1.047 (202 s)
epoch 6: loss 1.010 (243 s)
epoch 7: loss 0.989 (282 s)
epoch 8: loss 0.968 (322 s)
epoch 9: loss 0.951 (362 s)
epoch 10: loss 0.945 (402 s)
clean 78.7  linf 40.5  l2 62.3  l1 46.7  union 38.8   (30 s)


{'clean': 78.7, 'linf': 40.5, 'l2': 62.3, 'l1': 46.7, 'union': 38.8}

In [15]:
# same as my_loss(), but without l1 calculations at all

def my_loss_ablation(model, x, y):
    model.eval()
    x_linf = attack(model, x, y, "linf")
    x_l2 = attack(model, x, y, "l2")

    model.train()
    logits_linf = model(x_linf)
    logits_l2 = model(x_l2)

    loss_linf = F.cross_entropy(logits_linf, y)
    loss_l2 = F.cross_entropy(logits_l2, y)

    losses = torch.stack([loss_linf, loss_l2])
    weights = F.softmax(losses.detach(), dim=0)

    final_loss = weights[0] * loss_linf + weights[1] * loss_l2
    return final_loss

In [16]:
model_mine_ablation = adv_train(load_base_model(), my_loss_ablation)
test_model(model_mine_ablation)

epoch 1: loss 1.100 (26 s)
epoch 2: loss 1.067 (52 s)
epoch 3: loss 1.043 (78 s)
epoch 4: loss 1.013 (104 s)
epoch 5: loss 0.980 (130 s)
epoch 6: loss 0.958 (156 s)
epoch 7: loss 0.927 (182 s)
epoch 8: loss 0.911 (208 s)
epoch 9: loss 0.901 (234 s)
epoch 10: loss 0.897 (260 s)
clean 78.7  linf 42.5  l2 60.4  l1 26.9  union 26.5   (30 s)


{'clean': 78.7, 'linf': 42.5, 'l2': 60.4, 'l1': 26.9, 'union': 26.5}